In [1]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_silver_Object_Permission"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Bronze/All_Permissions" # ← Change source path
TARGET_PATH = "abfss://silver/object_permission" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_silver_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_silver_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_silver_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, 311999d2-2021-482b-afcd-8e9af80f71de, 3, Finished, Available, Finished)

🔧 Initializing ntk_silver_Object_Permission...
🚀 Starting ntk_silver_Object_Permission


In [2]:
source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Bronze_lakehouse.Lakehouse/Files/Bronze_layer/SharePointFiles"
target_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting"

StatementMeta(, 311999d2-2021-482b-afcd-8e9af80f71de, 4, Finished, Available, Finished)

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lit, current_timestamp, to_timestamp, when, upper, regexp_replace, split, explode
)
from pyspark.sql.types import StringType, IntegerType
from datetime import datetime

# Initialize Spark
spark = SparkSession.builder.appName("DimPermissionETL").getOrCreate()

# File Paths
today = datetime.now()  
from datetime import datetime, timedelta
today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d") 

levels_path = f"{source_path}/{year}/{month}/{day}/All_Permissions.csv"
levels_path_all = f"{source_path}/{year}/{month}/{day}/Permission_Levels.csv"

# Current Date for Path
current_date = datetime.now()
year, month, day = current_date.strftime("%Y"), current_date.strftime("%m"), current_date.strftime("%d")

output_path = f"{target_path}/{year}/{month}/{day}/object_permission.parquet"

# Functions
def add_metadata(df):
    return df.withColumn("SnapshotDate", current_timestamp()) \
             .withColumn("ProcessedDate", current_timestamp()) \
             .withColumn("DataSource", lit("SharePoint"))

def cast_all_to_string(df):
    for field in df.schema.fields:
        if field.name not in ["SnapshotDate", "ProcessedDate", "DataSource"]:
            df = df.withColumn(field.name, col(field.name).cast(StringType()))
    return df

def trim_columns(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, trim(col(field.name)))
    return df

def normalize_dates(df, columns):
    for col_name in columns:
        if col_name in df.columns:
            df = df.withColumn(col_name, to_timestamp(col(col_name)))
    return df

# Read CSV
df_prem = spark.read.option("header", True).csv(levels_path)
print(df_prem.count())

df_all = spark.read.option("header", True).csv(levels_path_all)
print(df_all.count())

df_prem.show(1)
df_all.show(2)


StatementMeta(, 311999d2-2021-482b-afcd-8e9af80f71de, 5, Finished, Available, Finished)

34
10
+--------------------+---------------+----------+-------------+--------------------+--------------------+------------+--------------------+--------------------+---------------+--------------------+----------------+---------------------+----------+---------------+--------------+-------------------+----------------+-----------------+-------------+--------------------+--------------------+--------------------+-----------+--------------------------+-------+-------+---------+----------+----------------+--------------------+---------------+----------------+---------------+---------------------+-------------------+----------------------+--------------------+-------------+--------------------+--------------------+---------------+----------+-------+-----------+-----------+---------------+------------+----------+-----------+----------------+---------+--------------+--------------------+-------------+
|        PermissionId|          Level|ObjectType|   ObjectName|           ObjectUrl|      

In [4]:


# ---------------------------
# Split & Explode PermissionLevel
# ---------------------------
df_prem = df_prem.withColumn("PermissionLevel", split(trim(col("PermissionLevel")), ";"))
df_exploded = df_prem.withColumn("PermissionLevel", explode(col("PermissionLevel")))
df_cleaned = df_exploded.withColumn("PermissionLevel", trim(col("PermissionLevel")))

# ---------------------------
# Add Metadata
# ---------------------------
df_etl = add_metadata(df_cleaned)

# ---------------------------
# Clean & Normalize
# ---------------------------
df_etl = cast_all_to_string(df_etl)
df_etl = trim_columns(df_etl)
df_etl = normalize_dates(df_etl, ["GrantedOn", "SnapshotDate", "ProcessedDate"])

# Replace Null-like values
null_values = ["", "NULL", "null", "N/A", "n/a"]
for val in null_values:
    df_etl = df_etl.replace(val, None)

# Show Sample
df_etl.show(2, truncate=False)
print(df_etl.count())
# print(f"Permission file saved to silver layer: {output_path}")


StatementMeta(, 311999d2-2021-482b-afcd-8e9af80f71de, 6, Finished, Available, Finished)

+------------------------------------+---------------+----------+-------------+------------------------------------------------------------+------------------------------------+------------+---------------------+---------------------+---------------+--------------------+----------------+---------------------+----------+---------------+--------------+-------------------+----------------+-----------------+---------------+--------------------+-----------------------------------------------+-----------------------------------------------------------------------------+-----------+--------------------------+-------+-------+---------------+--------------------+----------------+-----------------------------+---------------+----------------------------------+---------------+---------------------+-------------------+----------------------+--------------------+-------------+--------------------+---------------------+---------------+----------+-------+-----------+-----------+---------------+------

In [5]:
from pyspark.sql.functions import col, when, trim, sha2, coalesce, concat, lit

# Creating hash keys for Peremission and Objects

# Step 1: Create identifier column with email as priority, fallback to display name
df_etl = df_etl.withColumn(
    "User_Ident",
    when(
        # Check if email is not null and not empty/blank
        (col("PrincipalEmail").isNotNull()) & 
        (trim(col("PrincipalEmail")) != "") & 
        (col("PrincipalEmail") != " "),
        trim(col("PrincipalEmail"))  # Use email if valid
    ).otherwise(
        # Fallback to display name
        when(
            (col("PrincipalName").isNotNull()) & 
            (trim(col("PrincipalName")) != "") & 
            (col("PrincipalName") != " "),
            trim(col("PrincipalName"))
        ).otherwise(None)  # Set to null if both are blank
    )
)

# Creating UserKey & GroupKey column -As per  PrincipalType *****
df_etl = df_etl.withColumn(
    "UserKey",
    when(
        (col("PrincipalType") == "User") & (col("User_Ident").isNotNull()),
        sha2(col("User_Ident"), 256)
    ).otherwise(None)
)

# Creating GroupKey column - only for PrincipalType = "Group"  
df_etl = df_etl.withColumn(
    "GroupKey",
    when(
        (col("PrincipalType") != "User") & (col("User_Ident").isNotNull()),
        sha2(col("User_Ident"), 256)
    ).otherwise(None)
)

# Creating ObjectKey hashvalue
df_etl = df_etl.withColumn(
    "ObjectKey",
    when(
        col("ObjectUrl").isNotNull(),
        sha2(col("ObjectUrl"), 256)
    ).otherwise(None)
)

# Creating PermissionKey hashvalue
df_etl = df_etl.withColumn(
    "PermissionKey",
    when(
        col("PermissionLevel").isNotNull(),
        sha2(col("PermissionLevel"), 256)
    ).otherwise(None)
)

df_etl.show(5)
print(df_etl.count())
print(f"Userkey / Groupkey / PermissionKey and ObjectKey Created")

StatementMeta(, 311999d2-2021-482b-afcd-8e9af80f71de, 7, Finished, Available, Finished)

+--------------------+---------------+----------+-------------+--------------------+--------------------+------------+--------------------+--------------------+---------------+--------------------+----------------+---------------------+----------+---------------+--------------+-------------------+----------------+-----------------+---------------+--------------------+--------------------+--------------------+-----------+--------------------------+-------+-------+---------------+--------------------+----------------+--------------------+--------------------+--------------------+---------------+---------------------+-------------------+----------------------+--------------------+-------------+--------------------+--------------------+---------------+----------+-------+-----------+-----------+---------------+------------+----------+-----------+----------------+---------+--------------+--------------------+-------------+--------------------+--------------------+----------+-----------------

In [6]:
# Writing back to Silver leyer as file.
# df_etl.show(5, truncate=False)
print(df_etl.count())

df_etl.write.mode("overwrite").option("compression", "snappy").parquet(output_path)
# df_etl.coalesce(1).write.mode("overwrite").parquet(output_path)

# df_etl.printSchema()
print(f"Final data writed to : {output_path}")

StatementMeta(, 311999d2-2021-482b-afcd-8e9af80f71de, 8, Finished, Available, Finished)

42
Final data writed to : abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/15/object_permission.parquet


In [7]:
from datetime import datetime

# Define the log_etl_activity function for logging ETL process
def log_etl_activity(status, start_time, rows_read=None, rows_written=None, bytes_processed=None, error_details=None):

    end_time = datetime.now()
    duration_seconds = (end_time - start_time).total_seconds()

    log_message = {
        'Status': status,
        'StartTime': start_time,
        'EndTime': end_time,
        'DurationSeconds': duration_seconds,
        'RowsRead': rows_read,
        'RowsWritten': rows_written,
        'BytesProcessed': bytes_processed,
        'ErrorDetails': error_details
    }

    # For simplicity, let's print the log message (this can be replaced with a logging system)
    print("Logging ETL Activity:", log_message)

# Ensure processing_successful is defined before this block
try:
    NOTEBOOK_NAME = "ETL_Pipeline_Example"  # Define your notebook name or use the existing one
    start_time = datetime.now()  # Capture the start time of the ETL process

    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Simulated metrics
    rows_read = df_prem.count()
    rows_read = df_all.count()  # Correct this to have a meaningful `rows_read`
    rows_written = df_etl.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details=error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, 311999d2-2021-482b-afcd-8e9af80f71de, 9, Finished, Available, Finished)

🔄 Starting ETL processing for ETL_Pipeline_Example...
Logging ETL Activity: {'Status': 'SUCCESS', 'StartTime': datetime.datetime(2025, 10, 15, 5, 27, 19, 704199), 'EndTime': datetime.datetime(2025, 10, 15, 5, 27, 20, 747309), 'DurationSeconds': 1.04311, 'RowsRead': 10, 'RowsWritten': 42, 'BytesProcessed': 524288000, 'ErrorDetails': None}
🎉 ETL_Pipeline_Example pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 10 → 42
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ETL_Pipeline_Example:
+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ETL_Pipeline_Example logging completed!
